# Lab 5 — 캡스톤

> **회고** — S1 해부학 → S2 도구 → S3 루프 → S4 멀티에이전트·RAG
> 흩어진 부품이 아니라, 하나의 에이전트로 쌓아 올렸습니다.

## 목표
배운 것을 모아, 두 연구 데이터 위에서 **나만의 모빌리티 분석 에이전트**를
한 가지 이상 *확장* 합니다. 처음부터 만들지 않습니다 — 아래 재료를 출발점으로.


## 0. 가진 재료

```
common/agent.py   완성된 Agent 클래스
common/tools.py   분석 도구 5종 + 스키마 (tools.get_schemas / tools.TOOLBOX)
common/mini_sim.py 미니 택시 시뮬레이터
common/llm.py     LLMClient / MockLLM / ScriptedLLM
data/             regions · living_areas · od_flows · fleet_demand · knowledge
```


In [ ]:
from common.llm import LLMClient
from common.agent import Agent
from common import tools, mini_sim

# sanity check — the agent still works
agent = Agent(LLMClient(), tools.get_schemas(), dict(tools.TOOLBOX),
              system="You are a mobility analyst. Answer in Korean.")
print(agent.run("통근 통행 상위 3개 구간은?"))

## 1. 아이디어 카탈로그

- **★ 새 도구** — 두 지역/생활권 비교, 생활권 요약, 통행 통계 등
- **★★ 시뮬레이터 확장** — `mini_sim` 에 새 배차 전략을 더하고 도구로 노출
- **★★★ 구조 확장** — 새 서브에이전트, RAG 지식베이스 확장

## 2. 워크된 예시 — `compare_regions` 도구

새 도구 하나를 끝까지 만드는 모습입니다. 함수 → 스키마 → 에이전트에 등록.


In [ ]:
def compare_regions(region_a, region_b):
    """Compare two sigungu by their largest commute flows."""
    summary = {}
    for label, name in [("region_a", region_a), ("region_b", region_b)]:
        info = tools.get_region_flows(name, n=1)
        if "error" in info:
            return info
        summary[label] = {
            "region": info["region"],
            "living_area": info["living_area"],
            "biggest_outbound": info["top_outbound"],
            "biggest_inbound": info["top_inbound"],
        }
    return summary


COMPARE_SCHEMA = {
    "name": "compare_regions",
    "description": "Compare two sigungu side by side by their commute flows.",
    "parameters": {"type": "object", "properties": {
        "region_a": {"type": "string", "description": "First sigungu name."},
        "region_b": {"type": "string", "description": "Second sigungu name."},
    }, "required": ["region_a", "region_b"]},
}

capstone_agent = Agent(
    LLMClient(),
    schemas=tools.get_schemas() + [COMPARE_SCHEMA],
    functions={**tools.TOOLBOX, "compare_regions": compare_regions},
    system="You are a mobility analyst. Answer in Korean.",
)
print(capstone_agent.run("강남구와 분당구의 통근 패턴을 비교해줘."))

## 🔧 캡스톤 TODO — 나만의 확장

아래 스캐폴드를 **여러분의 아이디어로** 바꾸세요. 위 `compare_regions` 가 본보기입니다.
함수를 만들고, 스키마를 쓰고, 에이전트에 등록한 뒤, 질문으로 시연하세요.


In [ ]:
# 🔧 TODO: my_tool 을 여러분의 아이디어로 바꾸세요.
def my_tool(query: str) -> dict:
    """TODO: replace with your own tool logic."""
    return {"note": "not implemented yet", "query": query}


MY_SCHEMA = {
    "name": "my_tool",
    "description": "TODO: describe what your tool does.",
    "parameters": {"type": "object", "properties": {
        "query": {"type": "string", "description": "TODO"},
    }, "required": ["query"]},
}

my_agent = Agent(
    LLMClient(verbose=False),
    schemas=tools.get_schemas() + [MY_SCHEMA],
    functions={**tools.TOOLBOX, "my_tool": my_tool},
    system="You are a mobility analyst. Answer in Korean.",
)
print("스캐폴드 준비 완료 — my_tool 을 여러분의 아이디어로 채워보세요.")

## 발표 체크리스트 (1팀 5분)

1. **무엇을 만들었나** — 한 문장 + 데모 질문 1개
2. **어떻게 동작하나** — 도구 / 루프 / (있다면) 서브에이전트
3. **막혔던 점** — 어떤 한계를 만나 어떻게 풀었나
4. **라이브 시연** — 질문 → 추론·도구 호출 과정 보여주기

> 코드의 완성도보다, **개념을 제대로 썼는지**가 핵심입니다. 수고하셨습니다 🎉
